# Chapter 17 &mdash; Boolean Functions and Why Truth Tables Do Not Scale

**Concept 1 of the Chapter 17 decomposition:** *Boolean Functions, Truth-Table Personalities, and Why Tables Do Not Scale*

There are $2^{2^N}$ functions of $N$ inputs; a 64-input table would need more rows than atoms in a city.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Truth-Tables-Do-Not-Scale/Concept-Truth-Tables-Do-Not-Scale.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


An $N$-input Boolean function is fixed by its **truth table**: $2^N$ rows. So there are
$2^{2^N}$ distinct functions &mdash; 16 for $N=2$, 256 for $N=3$, and for $N=6$ already
more than $10^{19}$.

The table itself is the problem. At $N=64$ a truth table has $1.8\times10^{19}$ rows;
at one byte per row that is 18 exabytes.

Yet the functions we actually care about &mdash; adders, comparators, control logic &mdash;
have enormous **regularity**. The question the chapter answers is how to exploit it:
represent the function by a structure whose size tracks its **complexity** rather than
its **arity**.

The answer, surprisingly, is a **minimal DFA**.

## 2. Definitions

### Counting

In [ ]:
def n_functions(N): return 2 ** (2 ** N)
def table_rows(N):  return 2 ** N

def truth_table(f, N):
    from itertools import product
    return [(bits, f(bits)) for bits in product([0, 1], repeat=N)]

### Some regular functions, for contrast

In [ ]:
def AND_n(bits):  return int(all(bits))
def PARITY(bits): return sum(bits) % 2
def MAJORITY(bits): return int(sum(bits) * 2 > len(bits))

## 3. Tests

The double exponential.

In [ ]:
print("%-4s %-14s %s" % ("N", "table rows", "distinct functions"))
for N in [1, 2, 3, 4, 6]:
    print("%-4d %-14s %s" % (N, format(table_rows(N), ','), format(n_functions(N), ',')))
assert n_functions(6) > 10 ** 19

At $N=64$ the table is unusable.

In [ ]:
for N in [16, 32, 64]:
    rows = table_rows(N)
    print("  N=%2d : %s rows, %.3g bytes at one byte per row"
          % (N, format(rows, ','), float(rows)))
assert table_rows(64) > 1.8e19

But the functions we build hardware from are **regular**.

In [ ]:
for name, f in [('AND', AND_n), ('PARITY', PARITY), ('MAJORITY', MAJORITY)]:
    t = truth_table(f, 3)
    print("  %-9s :" % name, ''.join(str(v) for _, v in t))
print("\nEach is describable in a sentence -- the table is a bad encoding of that.")

Regularity shows up as **repeated subfunctions**.

In [ ]:
from itertools import product
def subfunctions(f, N):
    # fix the first variable; how many distinct residual functions are there?
    subs = set()
    for first in [0, 1]:
        subs.add(tuple(f((first,) + rest) for rest in product([0, 1], repeat=N - 1)))
    return subs
for name, f in [('AND', AND_n), ('PARITY', PARITY), ('MAJORITY', MAJORITY)]:
    print("  %-9s distinct residual functions after fixing x1 : %d"
          % (name, len(subfunctions(f, 4))))
print("\nTwo, always -- and that is what a BDD exploits.")

The plan for the chapter.

In [ ]:
print("truth table : size 2^N always")
print("BDD         : size tracks the number of DISTINCT residual functions")
print()
print("And 'distinct residual function' is exactly Myhill-Nerode's")
print("'distinguishable state'.  Concept 2 makes that identification.")

## 4. Exercises


1. How many 4-input functions are there? Write the number out.
2. Which 2-input functions are *not* expressible with AND, OR, NOT? (Trick question.)
3. Give a function of $N$ inputs with $2^{N-1}$ distinct residuals.

In [ ]:
# Your work for the exercises above.